# Clase 002 — Jupyter y JupyterLab

**Parte 0 — Prerrequisitos** · VanderPlas cap. 1.

> 🎯 Dejar de usar Jupyter como editor de texto y empezar a usarlo como entorno exploratorio: magics, debug interactivo, profiling.

> ⏱️ ~90 min

## 🗺️ Agenda

1. Kernel vs frontend vs servidor
2. Magics esenciales (`%timeit`, `%debug`, `%prun`)
3. Registrar un kernel propio
4. Ejercicio de profiling
5. Checklist + homework

## ⚙️ Setup

In [ ]:
import sys, time
import numpy as np
print('python:', sys.version.split()[0])
print('exec  :', sys.executable)
print('numpy :', np.__version__)

## 1️⃣ Magics esenciales

**Line magic** (`%`) afecta a la línea. **Cell magic** (`%%`) afecta a toda la celda.

- `%timeit expr` → microbenchmark con varias corridas (descarta outliers).
- `%%time` → tiempo total de la celda (una sola corrida).
- `%who` → lista variables del namespace.
- `%matplotlib inline` → gráficos dentro del notebook.
- `%load_ext autoreload` + `%autoreload 2` → recarga módulos sin reiniciar kernel.

In [ ]:
# Microbenchmark riguroso: sum nativo vs numpy
N = 100_000
l = list(range(N))
a = np.arange(N)

t0 = time.perf_counter(); s1 = sum(l); t1 = time.perf_counter()
t2 = time.perf_counter(); s2 = a.sum(); t3 = time.perf_counter()

print(f'sum(range): {(t1-t0)*1000:.3f} ms')
print(f'np.sum   : {(t3-t2)*1000:.3f} ms')
print(f'speedup  : {(t1-t0)/(t3-t2):.1f}x')
print('(En Jupyter, prefiere %timeit que repite la medición.)')

## 2️⃣ Debugging interactivo

Cuando una celda explota, NO la re-ejecutas a ciegas — usa `%debug` en la siguiente celda y entras al stack en el punto exacto del error.

Comandos pdb:
- `n` (next), `s` (step into), `c` (continue), `q` (quit)
- `p var` (print), `pp var` (pretty print)
- `u` / `d` (sube/baja en el stack)
- `l` (lista código alrededor)

In [ ]:
# Simulamos un bug típico — comenta la línea raise para evitar romper la ejecución del notebook
def divide_safe(a, b):
    return a / b  # bug: no valida b == 0

try:
    divide_safe(10, 0)
except ZeroDivisionError as e:
    print(f'ERROR: {e}')
    print('En Jupyter, ahora ejecutarías %debug en la siguiente celda para entrar al pdb post-mortem.')

## 3️⃣ Profiling — saber qué optimizar

Reglas:
1. **Mide antes de optimizar.** No adivines.
2. `%timeit` para microbenchmarks de una expresión.
3. `%prun` para perfil de función completa (tiempo por llamada).
4. `%lprun` (requiere `pip install line_profiler`) para perfil línea por línea.
5. `%memit` (requiere `memory_profiler`) para memoria.

In [ ]:
# Función deliberadamente ineficiente para profilar
def ordena_lento(xs):
    # Bubble sort O(n^2) — no hagan esto en prod
    xs = list(xs)
    n = len(xs)
    for i in range(n):
        for j in range(n - i - 1):
            if xs[j] > xs[j+1]:
                xs[j], xs[j+1] = xs[j+1], xs[j]
    return xs

import random
data = [random.random() for _ in range(500)]

t0 = time.perf_counter(); ordena_lento(data); t1 = time.perf_counter()
t2 = time.perf_counter(); sorted(data); t3 = time.perf_counter()
print(f'bubble : {(t1-t0)*1000:.2f} ms')
print(f'sorted : {(t3-t2)*1000:.2f} ms')
print(f'ratio  : {(t1-t0)/(t3-t2):.0f}x más lento')
print()
print('En Jupyter: %prun -s cumulative ordena_lento(data)')
print('te muestra dónde se gasta el tiempo, no solo cuánto.')

## 4️⃣ Registrar tu propio kernel

Evita el bug clásico de la clase 001 ("pip install funcionó pero import falla"):

```bash
# Desde tu venv activo:
python -m pip install ipykernel
python -m ipykernel install --user --name ds-lab --display-name 'DS Lab (mi venv)'
```

Luego en Jupyter: **Kernel → Change Kernel → DS Lab**. Verifica con `import sys; sys.executable` que apunta a tu `.venv`.

In [ ]:
# Verifica el kernel activo
import sys
from pathlib import Path

print('Kernel ejecutando este notebook:')
print(f'  {sys.executable}')
print()
in_venv = sys.prefix != sys.base_prefix
print(f'¿Es un venv?: {in_venv}')
if not in_venv:
    print('⚠️  Estás en el Python global. Registra un kernel propio antes de instalar paquetes.')

## ✅ Checklist

- [ ] Sé navegar Jupyter sin mouse (modo comando vs edición)
- [ ] Uso `%timeit` en vez de cronometrar a ojo
- [ ] Sé entrar a `%debug` cuando algo explota
- [ ] Mi notebook usa un kernel propio del venv del proyecto
- [ ] Sé qué función pesa más en mi código (con `%prun`)

## 📝 Homework

Ver `README.md` — entrega un notebook con benchmark `%timeit` comparando `sum(range(N))` vs `np.arange(N).sum()` para N=10k/100k/1M, tabla y gráfico.

## 📖 Definiciones y características

**Kernel**

Proceso Python (u otro lenguaje) que ejecuta el código de las celdas. Vive separado del frontend; si lo matas, pierdes el estado en memoria pero los archivos siguen intactos. Cada notebook se asocia a UN kernel, normalmente el del venv del proyecto.

**Frontend**

La interfaz visual (Notebook clásico, JupyterLab, VS Code, Cursor, Colab). Todas hablan el mismo protocolo con el kernel — puedes cambiar de frontend sin perder datos si guardas el `.ipynb`.

**Magic**

Comando especial de IPython, no de Python. Empieza con `%` (afecta una línea) o `%%` (afecta la celda entera). Ejemplos: `%timeit`, `%matplotlib inline`, `%%time`, `%debug`. No funcionan fuera de IPython/Jupyter.

**`%timeit` vs `%%time`**

`%timeit` corre la expresión **muchas veces**, descarta outliers y reporta el mejor → microbenchmark estadísticamente serio. `%%time` mide **una sola corrida** del bloque → bueno para operaciones largas donde repetir cuesta. Característica clave: usa `%timeit` para algo en milisegundos, `%%time` para algo en segundos.

**pdb / `%debug`**

Debugger interactivo de Python. `%debug` lo lanza en modo **post-mortem** después de una excepción — entras al stack en el punto del error sin re-correr nada. Comandos: `n` siguiente línea, `s` entra a función, `c` continúa, `p var` imprime, `u/d` sube/baja en stack, `q` salir.

## ⚠️ Errores comunes

| Síntoma / mensaje | Causa y cómo arreglar |
|---|---|
| `ModuleNotFoundError` aunque acabo de instalar el paquete | El kernel activo NO es el venv donde corriste `pip install`. **Fix**: en una celda, `import sys; print(sys.executable)` — si no apunta a tu venv, cambia el kernel (menú Kernel → Change Kernel) o registra el venv con `python -m ipykernel install --user --name <nombre>`. |
| El notebook está "congelado" / la barra dice `[*]` | Una celda quedó atrapada en bucle infinito o esperando input. **Fix**: menú Kernel → Interrupt (Esc + I dos veces). Si no responde, Restart Kernel — perderás variables en memoria pero los archivos quedan intactos. |
| Cambié código de un módulo importado y el notebook ignora el cambio | Python cachea módulos importados. **Fix**: `%load_ext autoreload` + `%autoreload 2` al inicio del notebook; recarga automáticamente al ejecutar. |
| `%timeit` en una celda con asignación da error "NameError" | Las variables creadas dentro de `%timeit` **no quedan** en el namespace (corre en sandbox). **Fix**: usa `%%timeit` (cell magic) si quieres preservar variables, o asigna fuera de la magic. |
| Outputs gigantes hacen el .ipynb pesado y el diff de git ilegible | Cada output (imagen, tabla) queda guardado en el JSON del notebook. **Fix**: pre-commit hook con `nbstripout` (limpia outputs antes de commitear) o `Cell → All Output → Clear` antes de guardar. |

## ❓ Preguntas frecuentes

**❓ ¿Notebook clásico o JupyterLab o VS Code?**

Para aprender, **VS Code** (mismo backend, mejor UX: autocomplete con type hints, debug gráfico, git inline). Para reuniones colaborativas en navegador, JupyterLab. El Notebook clásico es legacy — sigue funcionando pero ya no recibe features.

**❓ ¿Debo crear un kernel por proyecto o usar uno global?**

**Uno por proyecto.** Cada proyecto tiene dependencias distintas que entran en conflicto: el kernel global tarde o temprano se rompe. Comando: `python -m ipykernel install --user --name <proyecto>`.

**❓ ¿Cuándo `%timeit` no es confiable?**

Cuando lo que mides toca disco/red/GPU — la varianza es enorme y el min no representa típico. Usa `%%time` con varios runs manuales y reporta mediana. Tampoco confiable si la primera corrida hace JIT (numba) — calienta con un run previo.

**❓ `%debug` no funciona, no muestra prompt**

Necesita haber ocurrido una excepción **en el kernel** justo antes. Si la celda falló pero el kernel se reinició, perdiste el stack. También: en VS Code Jupyter, usa el panel de debug en su lugar (más cómodo).

**❓ ¿Por qué mi notebook tarda 30 segundos en abrir si pesa solo 200 KB?**

Probablemente trae outputs binarios grandes (imágenes inline en base64). El JSON parece chico pero al renderizar el navegador procesa MB. Limpia outputs y guarda.

## 🔗 Referencias

- VanderPlas, **cap. 1** — *IPython: Beyond Normal Python*
- [IPython magics](https://ipython.readthedocs.io/en/stable/interactive/magics.html)

➡️ **Siguiente:** [003 — Git y GitHub para data scientists](../003-git-y-github-para-data-scientists/README.md)

## ✅ Soluciones de los ejercicios

Intentá resolverlos vos primero; acá tenés una solución de referencia comentada. Algunos ejercicios usan
la UI de Jupyter (atajos) o magics interactivas (`%debug`) que no corren headless — abajo los **adaptamos**
a código autocontenido que demuestra el **mismo concepto** y corre sin intervención ni internet.

**Ejercicio 1.** Atajos sin mouse: crear celdas, convertir a markdown, ejecutar en orden, borrar una, deshacer.
(Adaptación: un `.ipynb` es JSON con una lista de celdas; replicamos esas operaciones sobre la lista con `nbformat`.)

In [ ]:
import nbformat
from nbformat.v4 import new_notebook, new_code_cell, new_markdown_cell

# Un notebook es una lista de celdas. Los atajos (A/B insertar, M markdown, Y codigo, X borrar) editan esta lista.
nbx = new_notebook()
nbx.cells = [new_code_cell(f'print({i})') for i in range(5)]   # 5 celdas de codigo (atajos: A/B)
print('Celdas iniciales :', len(nbx.cells), '->', [c.cell_type for c in nbx.cells])

# Convertir 2 a markdown (atajo: M)
for i in (0, 4):
    nbx.cells[i] = new_markdown_cell('# titulo')
print('Tras convertir 2 :', [c.cell_type for c in nbx.cells])

# Borrar una celda (atajo: D D  o  X)
borrada = nbx.cells.pop(2)
print('Tras borrar una  :', len(nbx.cells), 'celdas')

# Deshacer (atajo: Z) -> la reinsertamos
nbx.cells.insert(2, borrada)
print('Tras deshacer    :', len(nbx.cells), 'celdas')

assert len(nbx.cells) == 5
assert [c.cell_type for c in nbx.cells].count('markdown') == 2
print('\nOK: A/B insertan, M=markdown, Y=codigo, X=borrar, Z=deshacer. Practicalos sin tocar el mouse.')

**Ejercicio 2.** Registrá tu kernel y verificá con `sys.executable`. (Adaptación: listamos los kernels
registrados y confirmamos el intérprete activo, que es exactamente lo que hace la verificación.)

In [ ]:
import sys

print('Kernel activo (sys.executable):', sys.executable)

# Los kernels registrados se listan asi (equivalente a:  jupyter kernelspec list)
try:
    from jupyter_client.kernelspec import KernelSpecManager
    specs = KernelSpecManager().find_kernel_specs()
    print('\nKernels registrados en esta maquina:')
    for nombre, ruta in specs.items():
        print(f'   - {nombre}: {ruta}')
    assert len(specs) >= 1
except Exception as e:
    print('(No se pudo listar kernelspecs aqui:', type(e).__name__, '- no afecta el concepto)')

print('\nPara registrar el kernel de TU venv (en terminal, con el venv activo):')
print('    python -m ipykernel install --user --name ds-lab-001 --display-name "DS Lab 001"')
print('Luego en Jupyter eliges ese kernel y confirmas con  import sys; sys.executable')

**Ejercicio 3.** Benchmark de vectorización: `for` sobre `range` vs `np.arange().sum()`.
(Usamos el módulo `timeit` para que sea determinista headless; `%timeit` da lo mismo en Jupyter.)

In [ ]:
import timeit, numpy as np

N = 100_000
setup_np = 'import numpy as np; a = np.arange(N)'
def suma_for():
    total = 0
    for x in range(N):
        total += x
    return total

t_for = timeit.timeit(suma_for, number=20) / 20
t_np = timeit.timeit('a.sum()', setup=setup_np, number=200, globals={'N': N}) / 200

print(f'for + range  : {t_for*1e3:9.3f} ms')
print(f'np.arange.sum: {t_np*1e3:9.3f} ms')
speedup = t_for / t_np
print(f'\nNumPy es ~{speedup:.0f}x mas rapido para N={N:,}')

# Verificamos que ambos dan el MISMO resultado numerico
assert suma_for() == int(np.arange(N).sum())
assert speedup > 1, 'NumPy vectorizado deberia ganarle al bucle Python'
print('OK: vectorizar (NumPy en C) evita el overhead del bucle interpretado.')

**Ejercicio 4.** Post-mortem de un `ZeroDivisionError` con `%debug`. (Adaptación: `%debug` es interactivo
y colgaría un run headless; hacemos la **misma inspección post-mortem** leyendo el traceback y las variables
locales del frame donde explotó — que es justo lo que `%debug` te deja hacer a mano.)

In [ ]:
import sys

def dividir(a, b):
    resultado = a / b        # aqui explota cuando b == 0
    return resultado

capturada = None
try:
    dividir(10, 0)
except ZeroDivisionError as e:
    capturada = e
    tb = sys.exc_info()[2]
    # Bajamos al ultimo frame (donde ocurrio el error) -> equivalente a lo que ves con %debug
    frame = tb.tb_frame
    while tb.tb_next:
        tb = tb.tb_next
        frame = tb.tb_frame
    print('Excepcion   :', type(capturada).__name__, '-', capturada)
    print('Funcion     :', frame.f_code.co_name)
    print('Linea       :', tb.tb_lineno)
    print('Variables locales en el momento del error (p a / p b en pdb):')
    for k, v in frame.f_locals.items():
        print(f'    {k} = {v!r}')

assert isinstance(capturada, ZeroDivisionError)
print('\nOK: post-mortem = inspeccionar el estado JUSTO en el punto del fallo. En Jupyter: corre  %debug  tras el error.')

**Ejercicio 5.** Profilá una función lenta (bubble sort repetido) e identificá la parte más cara.
(Usamos `cProfile` + `pstats`, que es lo que hace `%prun` por debajo.)

In [ ]:
import cProfile, pstats, io, random

def bubble_sort(xs):
    xs = xs[:]
    n = len(xs)
    for i in range(n):
        for j in range(n - 1 - i):
            if xs[j] > xs[j + 1]:
                xs[j], xs[j + 1] = xs[j + 1], xs[j]
    return xs

def correr_muchas_veces():
    base = [random.randint(0, 999) for _ in range(60)]
    for _ in range(300):
        bubble_sort(base)

prof = cProfile.Profile()
prof.enable()
correr_muchas_veces()
prof.disable()

s = io.StringIO()
pstats.Stats(prof, stream=s).sort_stats('cumulative').print_stats(5)
salida = s.getvalue()
print(salida)

assert 'bubble_sort' in salida, 'El profiler deberia mostrar bubble_sort como la funcion cara'
print('OK: bubble_sort domina el tiempo (bucle anidado O(n^2)). En Jupyter:  %prun -s cumulative correr_muchas_veces()')